In [5]:
!pip install torch_geometric

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

# Load the MUTAG dataset (molecule classification)
dataset = TUDataset(root='/tmp/MUTAG', name='MUTAG')
# Shuffle and split the dataset (80% training and 20% testing)
torch.manual_seed(42)
dataset = dataset.shuffle()
train_dataset = dataset[:int(len(dataset) * 0.8)]
test_dataset = dataset[int(len(dataset) * 0.8):]

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Define a GCN-based Graph Classification Model
class GCNClassifier(nn.Module):
    def __init__(self, num_features, hidden_dim, num_classes):
        super(GCNClassifier, self).__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        # A fully connected layer for final classification
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # First GCN layer with ReLU activation
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        # Second GCN layer with ReLU activation
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Global mean pooling to obtain graph-level embeddings
        x = global_mean_pool(x, batch)

        # Final classification layer
        x = self.fc(x)
        return F.log_softmax(x, dim=1)

# Set device, hyperparameters, and instantiate model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCNClassifier(num_features=dataset.num_features, hidden_dim=64, num_classes=dataset.num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Training function
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = F.nll_loss(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(train_dataset)

# Testing function
def test(loader):
    model.eval()
    correct = 0
    for data in loader:
        data = data.to(device)
        out = model(data)
        pred = out.max(1)[1]
        correct += pred.eq(data.y).sum().item()
    return correct / len(loader.dataset)

# Training Loop
for epoch in range(1, 201):
    loss = train()
    train_acc = test(train_loader)
    test_acc = test(test_loader)
    if epoch % 10 == 0:
        print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")


Epoch: 010, Loss: 0.5327, Train Acc: 0.7733, Test Acc: 0.6316
Epoch: 020, Loss: 0.4924, Train Acc: 0.7800, Test Acc: 0.6842
Epoch: 030, Loss: 0.4943, Train Acc: 0.7667, Test Acc: 0.6842
Epoch: 040, Loss: 0.4868, Train Acc: 0.7733, Test Acc: 0.6579
Epoch: 050, Loss: 0.4813, Train Acc: 0.7667, Test Acc: 0.6842
Epoch: 060, Loss: 0.4653, Train Acc: 0.7667, Test Acc: 0.6579
Epoch: 070, Loss: 0.4753, Train Acc: 0.7533, Test Acc: 0.6579
Epoch: 080, Loss: 0.4633, Train Acc: 0.7667, Test Acc: 0.6842
Epoch: 090, Loss: 0.4679, Train Acc: 0.7533, Test Acc: 0.6579
Epoch: 100, Loss: 0.4650, Train Acc: 0.7533, Test Acc: 0.6842
Epoch: 110, Loss: 0.4685, Train Acc: 0.7533, Test Acc: 0.6842
Epoch: 120, Loss: 0.4626, Train Acc: 0.7467, Test Acc: 0.6842
Epoch: 130, Loss: 0.4569, Train Acc: 0.7867, Test Acc: 0.7105
Epoch: 140, Loss: 0.4636, Train Acc: 0.7467, Test Acc: 0.6842
Epoch: 150, Loss: 0.4606, Train Acc: 0.7533, Test Acc: 0.6842
Epoch: 160, Loss: 0.4627, Train Acc: 0.7533, Test Acc: 0.6842
Epoch: 1